In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

In [ ]:
df = pd.read_csv("enriched_incidents.csv")

<center> Count Plots

In [ ]:
sns.countplot(data=df, x="Event_Severity", order=df["Event_Severity"].value_counts().index)
plt.xticks(rotation=45, ha="right")
plt.show()
sns.countplot(data=df, x="Restraint_SubType", order=df["Restraint_SubType"].value_counts().index)
plt.xticks(rotation=45, ha="right")
plt.show()
sns.countplot(data=df, x=pd.cut(df["Age"], bins=[0,25,40,60,100]))
plt.show()
sns.countplot(data=df, x="Gender")
plt.show()
top_eth = df["Ethnicity"].value_counts().nlargest(5).index
df["Ethnicity_plot"] = df["Ethnicity"].where(df["Ethnicity"].isin(top_eth), "Other")
sns.countplot(data=df, x="Ethnicity_plot")
plt.xticks(rotation=45, ha="right")
plt.show()
sns.countplot(data=df, x="on_antipsychotic_24h")
plt.show()
sns.countplot(data=df, x="num_prn_72h")
plt.show()
sns.countplot(data=df, x="Event_Severity", hue="Gender")
plt.xticks(rotation=45, ha="right")
plt.show()
sns.countplot(data=df, x="Restraint_SubType", hue="Event_Severity")
plt.xticks(rotation=45, ha="right")
plt.show() 
sns.countplot(data=df, x="MH_Section", hue="Seclusion")
plt.xticks(rotation=45, ha="right")
plt.show()



In [ ]:
df["Time_Stamp"] = pd.to_datetime(df["Time_Stamp"], errors="coerce")

incidents_per_patient = df.groupby("Patient_ID")["Time_Stamp"].count()

plt.figure(figsize=(8,6))
plt.hist(incidents_per_patient, bins=30, edgecolor="black")
plt.title("Histogram of Incidents per Patient")
plt.xlabel("Number of Incidents")
plt.ylabel("Number of Patients")
plt.grid(False)  
plt.show()
plt.figure(figsize=(8,6))
plt.hist(incidents_per_patient, bins=30, edgecolor="black", log=True)
plt.title("Histogram of Incidents per Patient (Log Scale)")
plt.xlabel("Number of Incidents")
plt.ylabel("Number of Patients (log scale)")
plt.grid(False)
plt.show()


In [ ]:
plt.figure(figsize=(10,6))
sns.countplot(
    data=df, 
    x="Ethnicity", 
    order=df["Ethnicity"].value_counts().index,
    palette="viridis"
)

plt.title("Distribution of Ethnicity in Dataset", fontsize=14)
plt.xlabel("Ethnicity")
plt.ylabel("Count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6,5))
sns.countplot(
    data=df,
    x="Gender",
    order=df["Gender"].value_counts().index,
    palette="Set2"
)

plt.title("Distribution of Gender in Dataset", fontsize=14)
plt.xlabel("Gender")
plt.ylabel("Count")

total = len(df)
for p in plt.gca().patches:
    count = int(p.get_height())
    percentage = f"{100 * count / total:.1f}%"
    plt.gca().annotate(
        percentage,
        (p.get_x() + p.get_width() / 2., count),
        ha='center', va='bottom',
        fontsize=10, color='black'
    )

plt.show()


In [ ]:
plt.figure(figsize=(12,6))
sns.countplot(
    data=df,
    y="Diagnosis",
    order=df["Diagnosis"].value_counts().head(15).index,  # top 15 diagnoses
    palette="tab10"
)

plt.title("Distribution of Diagnoses (Top 15)", fontsize=14)
plt.xlabel("Count")
plt.ylabel("Diagnosis")
plt.tight_layout()
plt.show()


<center> Other Data Viz

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from textwrap import fill

sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)


def topN_with_other(series, top_n=10, other_label="Other"):
    s = series.fillna("Missing")
    counts = s.value_counts()
    if len(counts) <= top_n:
        return s
    keep = counts.index[:top_n]
    return s.where(s.isin(keep), other_label)

def wrap_axis_labels(ax, width=14, axis="x"):
    labels = ax.get_xticklabels() if axis == "x" else ax.get_yticklabels()
    new_labels = [fill(lbl.get_text(), width=width) for lbl in labels]
    if axis == "x":
        ax.set_xticklabels(new_labels, rotation=0, ha="center")
    else:
        ax.set_yticklabels(new_labels)

def count_bar(df, col, title=None, top_n=None, horizontal=False, wrap_width=14):
    ser = df[col].fillna("Missing")
    if top_n is not None:
        ser = topN_with_other(ser, top_n=top_n)
    order = ser.value_counts().index
    ax = None
    if horizontal:
        ax = sns.countplot(data=df.assign(_tmp=ser), y="_tmp", order=order)
        ax.set_ylabel(col)
        wrap_axis_labels(ax, width=wrap_width, axis="y")
    else:
        ax = sns.countplot(data=df.assign(_tmp=ser), x="_tmp", order=order)
        ax.set_xlabel(col)
        wrap_axis_labels(ax, width=wrap_width, axis="x")
        plt.xticks(rotation=0, ha="center")
    ax.set_title(title or f"Counts of {col}")
    for c in ax.containers:
        ax.bar_label(c, fmt="%d", label_type="edge", padding=2)
    plt.tight_layout()
    plt.show()

def binary_bar(df, col, title=None):
    tmp = df[col].fillna(0).astype(int)
    ax = sns.countplot(x=tmp)
    ax.set_xlabel(col + " (0/1)")
    ax.set_title(title or f"Counts of {col}")
    for c in ax.containers:
        ax.bar_label(c, fmt="%d", label_type="edge", padding=2)
    plt.tight_layout()
    plt.show()

def stacked_severity_by_flags(df, severity_col, flags):
    d = df.copy()
    d[severity_col] = d[severity_col].fillna("Missing")
    for f in flags:
        if f not in d.columns:
            d[f] = 0
    long = d.melt(id_vars=[severity_col], value_vars=flags, var_name="Intervention", value_name="value")
    long = long[long["value"] == 1]
    counts = long.value_counts([severity_col, "Intervention"]).rename("count").reset_index()
    pivot = counts.pivot(index=severity_col, columns="Intervention", values="count").fillna(0).astype(int)
    pivot = pivot.loc[pivot.sum(axis=1).sort_values(ascending=False).index]

    ax = pivot.plot(kind="bar", stacked=True, figsize=(12, 6))
    ax.set_xlabel(severity_col)
    ax.set_ylabel("Incident count")
    ax.set_title("Incidents by Event_Severity, split by intervention type (stacked)")
    wrap_axis_labels(ax, width=16, axis="x")
    plt.tight_layout()
    plt.show()

def safe_boxplot(df, x, y, title, order=None, wrap_width=14):
    d = df[[x, y]].copy()
    d[x] = d[x].fillna("Missing")
    ax = sns.boxplot(data=d, x=x, y=y, order=order)
    ax.set_title(title)
    wrap_axis_labels(ax, width=wrap_width, axis="x")
    plt.tight_layout()
    plt.show()


if "Event_Severity" in inc.columns:
    count_bar(inc, "Event_Severity", title="Incidents by Event_Severity", horizontal=False, wrap_width=16)

for col in ["Seclusion", "Restraint", "Rapid_Tranq", "SelfHarm", "Safeguarding"]:
    if col in inc.columns:
        binary_bar(inc, col, title=f"{col} (0/1)")

if "Restraint_SubType" in inc.columns:
    count_bar(inc, "Restraint_SubType", title="Restraint subtypes (top 15)", top_n=15, horizontal=True, wrap_width=20)
if "Prone_Restraint" in inc.columns:
    binary_bar(inc, "Prone_Restraint", title="Prone_Restraint (0/1)")

if "Gender" in inc.columns:
    count_bar(inc, "Gender", title="Incidents by Gender", horizontal=False)

if "Ethnicity" in inc.columns:
    count_bar(inc, "Ethnicity", title="Incidents by Ethnicity (top 10 + Other)", top_n=10, horizontal=True, wrap_width=20)

if "Diagnosis" in inc.columns:
    count_bar(inc, "Diagnosis", title="Incidents by Diagnosis (top 12 + Other)", top_n=12, horizontal=True, wrap_width=26)


if "MH_Section" in inc.columns:
    count_bar(inc, "MH_Section", title="Incidents by MH_Section", top_n=15, horizontal=True, wrap_width=22)

if "on_antipsychotic_24h" in inc.columns:
    binary_bar(inc, "on_antipsychotic_24h", title="On antipsychotic in last 24h (0/1)")


if {"Patient_ID", "Days_Since_Admission"}.issubset(inc.columns):
    first_inc = inc.sort_values(["Patient_ID", "Days_Since_Admission"]).groupby("Patient_ID", as_index=False).first()
    ax = sns.histplot(first_inc["Days_Since_Admission"].dropna(), bins=40)
    ax.set_title("Days Since Admission at First Incident (per patient)")
    ax.set_xlabel("Days_Since_Admission (first incident)")
    plt.tight_layout()
    plt.show()

if "time_since_last_activity" in inc.columns:
    ax = sns.histplot(inc["time_since_last_activity"].dropna(), bins=40)
    ax.set_title("Time since last activity (hours) at incidents")
    ax.set_xlabel("Hours")
    plt.tight_layout()
    plt.show()

    mean_tsla = inc["time_since_last_activity"].mean()
    print(f"Average time_since_last_activity (hours): {mean_tsla:0.2f}")

if "Patient_ID" in inc.columns:
    counts = inc["Patient_ID"].value_counts()
    ax = sns.histplot(counts, bins=30)
    ax.set_title("Incident count per patient")
    ax.set_xlabel("Incidents per patient")
    plt.tight_layout()
    plt.show()


if "Event_Severity" in inc.columns:
    flags = [c for c in ["Restraint", "Seclusion", "Rapid_Tranq", "SelfHarm", "Safeguarding"] if c in inc.columns]
    if flags:
        stacked_severity_by_flags(inc, "Event_Severity", flags)


if "event_duration" in inc.columns:
    # by severity
    safe_boxplot(inc, x="Event_Severity", y="event_duration",
                 title="Event duration by severity", wrap_width=16)
    # by intervention 
    if "Restraint" in inc.columns:
        d = inc.copy()
        d["Restraint"] = d["Restraint"].fillna(0).astype(int).map({0: "No restraint", 1: "Restraint"})
        safe_boxplot(d, x="Restraint", y="event_duration",
                     title="Event duration by restraint")


if {"Age", "Event_Severity"}.issubset(inc.columns):
    ax = sns.violinplot(data=inc.dropna(subset=["Age"]), x="Event_Severity", y="Age", cut=0, inner="quart")
    ax.set_title("Age distribution by Event_Severity")
    wrap_axis_labels(ax, width=16, axis="x")
    plt.tight_layout()
    plt.show()


if {"Event_Severity", "Restraint_SubType"}.issubset(inc.columns):
    ct = pd.crosstab(inc["Event_Severity"].fillna("Missing"),
                     inc["Restraint_SubType"].fillna("Missing"))
    plt.figure(figsize=(12, 7))
    ax = sns.heatmap(ct, annot=False, fmt="d")
    ax.set_title("Event_Severity × Restraint_SubType (counts)")
    plt.tight_layout()
    plt.show()

flags_all = [c for c in ["Seclusion","Restraint","Prone_Restraint","Rapid_Tranq","SelfHarm",
                         "Safeguarding","Medi_Error","Death","on_antipsychotic_24h"]
             if c in inc.columns]
if flags_all:
    M = inc[flags_all].fillna(0).astype(float).corr()
    plt.figure(figsize=(8, 6))
    ax = sns.heatmap(M, vmin=-1, vmax=1, center=0, annot=True, fmt=".2f", linewidths=.5)
    ax.set_title("Correlation among intervention / outcome flags")
    plt.tight_layout()
    plt.show()

if "Days_Since_Admission" in inc.columns:
    dsa_counts = inc["Days_Since_Admission"].dropna().astype(int).value_counts().sort_index()
    ax = dsa_counts.plot(kind="line")
    ax.set_title("Incident counts by Days Since Admission")
    ax.set_xlabel("Days_Since_Admission")
    ax.set_ylabel("Incident count")
    plt.tight_layout()
    plt.show()


if "Time_Stamp" in inc.columns:
    ts_ok = inc.dropna(subset=["Time_Stamp"]).set_index("Time_Stamp")
    if not ts_ok.empty:
        weekly = ts_ok.resample("W").size()
        monthly = ts_ok.resample("MS").size()

        ax = weekly.plot()
        ax.set_title("Weekly incident frequency")
        ax.set_ylabel("Count")
        plt.tight_layout()
        plt.show()

        ax = monthly.plot()
        ax.set_title("Monthly incident frequency")
        ax.set_ylabel("Count")
        plt.tight_layout()
        plt.show()


In [ ]:
import missingno as msno

plt.figure(figsize=(12,6))
msno.matrix(df, sparkline=False)
plt.title("Missingness Map of Dataset", fontsize=14)
plt.show()


In [ ]:
from lifelines import KaplanMeierFitter

incident_times = (
    df.groupby("Patient_ID")["Days_Since_Admission"]
      .min()
      .reset_index()
      .rename(columns={"Days_Since_Admission": "time_to_first_incident"})
)

incident_times["event_observed"] = 1

kmf = KaplanMeierFitter()
kmf.fit(incident_times["time_to_first_incident"], incident_times["event_observed"])

plt.figure(figsize=(10,6))
kmf.plot_survival_function()
plt.title("Cumulative Incidents (Time to First Incident)", fontsize=14)
plt.xlabel("Days Since Admission")
plt.ylabel("Proportion Without Incident")
plt.show()


In [ ]:
df_sample = inc.sample(n=5000, random_state=42).drop(columns=["Patient_ID"], errors="ignore")

high_cardinality = ["Diagnosis", "Ethnicity"]
df_sample = df_sample.drop(columns=[c for c in high_cardinality if c in df_sample.columns])

categorical_cols = df_sample.select_dtypes(include=["object", "category"]).columns
df_encoded = pd.get_dummies(df_sample, columns=categorical_cols, drop_first=True, dummy_na=True)

corr = df_encoded.corr()

plt.figure(figsize=(12, 9))
sns.heatmap(corr, cmap="coolwarm", center=0, vmin=-1, vmax=1, cbar_kws={'label': 'Correlation'})
plt.title("Correlation Matrix (sampled, Patient_ID/High-cardinality excluded)")
plt.tight_layout()
plt.show()


In [ ]:
threshold = 0.2
corr_filtered = corr.copy()
corr_filtered[corr_filtered.abs() < threshold] = 0

plt.figure(figsize=(12, 9))
sns.heatmap(
    corr_filtered,
    cmap="coolwarm",
    center=0,
    vmin=-1, vmax=1,
    cbar_kws={'label': 'Correlation'},
    linewidths=0.5
)
plt.xticks(rotation=90, fontsize=8)
plt.yticks(fontsize=8)
plt.title(f"Correlation Matrix (|r| > {threshold})", fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
from IPython.display import display

def top_correlations(corr, n=20, exclude_trivial=True):
    corr_unstack = corr.where(~np.tril(np.ones(corr.shape), -1).astype(bool)) \
                       .stack() \
                       .reset_index()
    corr_unstack.columns = ["Variable 1", "Variable 2", "Correlation"]

    corr_unstack = corr_unstack[corr_unstack["Variable 1"] != corr_unstack["Variable 2"]]

    corr_unstack["|Correlation|"] = corr_unstack["Correlation"].abs()

    corr_sorted = corr_unstack.sort_values("|Correlation|", ascending=False)

    if exclude_trivial:
        drop_mask = corr_sorted.apply(
            lambda row: row["Variable 1"].split("_")[0] == row["Variable 2"].split("_")[0],
            axis=1
        )
        corr_sorted = corr_sorted[~drop_mask]

    top_corrs = corr_sorted.head(n).reset_index(drop=True)

    display(top_corrs.style.background_gradient(cmap="coolwarm", subset=["Correlation"]))

    return top_corrs

top_corrs = top_correlations(corr, n=20, exclude_trivial=True)
